# PlagIA Empresarial · 01 — Gobernanza de Datos, Auditoría de Fuga y Manifiesto Multinivel

**Subsistema de Ingeniería de Datos y MLOps (`engine`)**

### Objetivos Técnicos
1. Construir y cargar el **manifiesto unificado** para capturas de drones (`01_dron_dosel_cls`) y dispositivos móviles en terreno (`02_campo_hoja_cls`).
2. Auditar la integridad de datos (`auditar_fuga_datos` y `obtener_resumen_curaduria`) verificando cero fuga entre particiones (`entrenamiento`, `validacion`, `prueba`).
3. Ejecutar la ingesta distribuida con **PySpark (`construir_manifiesto_spark`)** preparada para Data Lakes.
4. Generar el subconjunto balanceado y el particionado **70% Entrenamiento / 15% Validación / 15% Prueba** agrupado por `group_id`.

In [ ]:
import sys
from pathlib import Path

RAIZ_PROYECTO = Path.cwd().resolve()
while not (RAIZ_PROYECTO / "pyproject.toml").exists() and RAIZ_PROYECTO != RAIZ_PROYECTO.parent:
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from engine.config import (
    DIR_CURADO_DRON,
    MUESTRAS_POR_CLASE,
    RUTA_MANIFIESTO_MULTINIVEL,
    RUTA_MANIFIESTO_PARTICIONADO,
    SEMILLA,
)
from engine.data_governance import (
    auditar_fuga_datos,
    cargar_manifiesto_dominio_curado,
    construir_manifiesto_dron,
    construir_manifiesto_multinivel,
    crear_subconjunto_balanceado,
    obtener_resumen_curaduria,
    particionar_manifiesto_agrupado,
)

print("Raíz del monorepo PlagIA:", RAIZ_PROYECTO)
print("Dominio Dron VANT       :", DIR_CURADO_DRON)

## 1. Auditoría de Calidad de Datos y Prevención de Fuga

In [ ]:
reporte_auditoria = auditar_fuga_datos(DIR_CURADO_DRON)
for clave, valor in reporte_auditoria.items():
    print(f"{clave:32s}: {valor}")

curaduria = obtener_resumen_curaduria()
print("\nClases Dron (Dominio 1)      :", curaduria["dominio_01_dron_dosel"]["clases"])
print("Clases Campo Hoja (Dominio 2):", curaduria["dominio_02_campo_hoja"]["clases"])

## 2. Construcción del Manifiesto Limpio (Dron VANT y Campo Hoja)

In [ ]:
manifiesto_dron = construir_manifiesto_dron(DIR_CURADO_DRON, excluir_copias=True)
manifiesto_hoja = cargar_manifiesto_dominio_curado("campo_hoja")
manifiesto_multinivel = construir_manifiesto_multinivel()

print(f"Total registros Dron VANT   : {len(manifiesto_dron)}")
print(f"Total registros Campo Hoja  : {len(manifiesto_hoja)}")
print("\nDistribución por clase (Dron VANT):")
print(manifiesto_dron["label_name"].value_counts())
manifiesto_dron.head()

## 3. Ingesta Distribuida con PySpark (Escalabilidad sobre Data Lake)

In [ ]:
try:
    from engine.data_governance import construir_manifiesto_spark
    sdf = construir_manifiesto_spark(DIR_CURADO_DRON)
    print("Total registros leídos con PySpark:", sdf.count())
    sdf.groupBy("label_name").count().show()
except Exception as exc:
    print(f"Aviso PySpark local ({type(exc).__name__}): se continúa con el manifiesto Pandas validado.")

## 4. Muestreo Balanceado y Partición 70/15/15 Agrupada sin Fuga de Datos

In [ ]:
df_subconjunto = crear_subconjunto_balanceado(manifiesto_dron, por_clase=MUESTRAS_POR_CLASE, semilla=SEMILLA)
df_particionado = particionar_manifiesto_agrupado(df_subconjunto, semilla=SEMILLA)

chequeo_fuga = df_particionado.groupby("group_id")["particion"].nunique()
grupos_filtrados = int((chequeo_fuga > 1).sum())
print("Grupos compartidos entre múltiples particiones (debe ser 0):", grupos_filtrados)
assert grupos_filtrados == 0, "Fuga de datos detectada en la partición"

print("\nComposición de la Partición 70/15/15:")
print(df_particionado.groupby(["particion", "label_name"]).size())

RUTA_MANIFIESTO_MULTINIVEL.parent.mkdir(parents=True, exist_ok=True)
manifiesto_multinivel.to_csv(RUTA_MANIFIESTO_MULTINIVEL, index=False)
df_particionado.to_csv(RUTA_MANIFIESTO_PARTICIONADO, index=False)
print("\nManifiestos guardados en:", RUTA_MANIFIESTO_PARTICIONADO)